In [2]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [3]:
target = "income-class"
sensitive_cols = ["marital-status", "relationship", "race", "sex", "native-country"]

In [4]:
data = pd.read_csv("adult.data.csv", na_values="?", skipinitialspace=True)

feature_cols = [c for c in data.columns if c not in [target] + sensitive_cols]
X = data[feature_cols]
y = data[target]
X_sensitive = data[sensitive_cols]

X_train, X_test, y_train, y_test, _, X_sensitivefeatures_test = train_test_split(
    X, y, X_sensitive, test_size=0.20, random_state=42, stratify=y
)

In [5]:
numeric_cols = X_train.select_dtypes(include=["number"]).columns.tolist()
categorical_cols = X_train.select_dtypes(exclude=["number"]).columns.tolist()

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocess = ColumnTransformer([
    ("num", numeric_pipeline, numeric_cols),
    ("cat", categorical_pipeline, categorical_cols),
])

In [ ]:
pipeline = Pipeline([
    ("preprocess", preprocess),
    ("knn", KNeighborsClassifier()),
])

param_grid = {
    "knn__n_neighbors": list(range(3, 42, 2)),
    "knn__weights": ["uniform", "distance"],
    "knn__p": [1, 2],
    "knn__leaf_size": [20, 30, 40],
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring="f1_macro",
    cv=cv,
    n_jobs=-1,
    verbose=1,
)

search.fit(X_train, y_train)
classifier = search.best_estimator_
y_predict = classifier.predict(X_test)

print("Best params:", search.best_params_)
print("Best CV f1_macro:", round(search.best_score_, 4))
print(confusion_matrix(y_test, y_predict))
print(classification_report(y_test, y_predict))

In [6]:
classifier = KNeighborsClassifier(
    n_neighbors=17,
    weights="uniform",
    leaf_size=20,
    p=2,
)

classifierPipeline = Pipeline([
    ("preprocess", preprocess),
    ("knn", classifier),
])

classifierPipeline.fit(X_train, y_train)
y_predict = classifierPipeline.predict(X_test)  # <- use pipeline here

print(confusion_matrix(y_test, y_predict))
print(classification_report(y_test, y_predict))

[[4615  330]
 [ 860  708]]
              precision    recall  f1-score   support

       <=50K       0.84      0.93      0.89      4945
        >50K       0.68      0.45      0.54      1568

    accuracy                           0.82      6513
   macro avg       0.76      0.69      0.71      6513
weighted avg       0.80      0.82      0.80      6513



## Individual fairness measurement (minimal)
Measures local individual-fairness violations for the fitted KNN pipeline using probability outputs and a normalized distance metric in the preprocessed feature space.

In [7]:
import numpy as np
from sklearn.neighbors import NearestNeighbors

# Prefer the fitted pipeline from the final evaluation cell; fall back to the fitted grid-search pipeline.
if "classifierPipeline" in globals():
    fairness_model = classifierPipeline
elif "classifier" in globals() and hasattr(classifier, "named_steps"):
    fairness_model = classifier
else:
    fairness_model = search.best_estimator_

fairness_cfg = {
    "positive_label": ">50K",
    "k_local": 15,                     # local neighborhood size (excluding self)
    "distance_scale_percentile": 95.0, # scales distances into [0, 1]
    "tau": 0.35,                       # local fairness threshold on normalized distance
}

# 1) Probability outputs p(x)
proba = fairness_model.predict_proba(X_test)
classes = list(fairness_model.classes_)
pos_idx = classes.index(fairness_cfg["positive_label"])
p = proba[:, pos_idx]

# 2) Baseline fairness metric d(x, y): normalized Euclidean distance in preprocessed (non-sensitive) feature space
X_test_preprocessed = fairness_model.named_steps["preprocess"].transform(X_test)

n_samples = X_test_preprocessed.shape[0]
n_neighbors = min(fairness_cfg["k_local"] + 1, n_samples)
if n_neighbors < 2:
    raise ValueError("Not enough samples to compute local fairness pairs.")

nn = NearestNeighbors(n_neighbors=n_neighbors, metric="euclidean")
nn.fit(X_test_preprocessed)
distances, indices = nn.kneighbors(X_test_preprocessed, return_distance=True)

# Drop self-neighbor (distance 0)
local_distances = distances[:, 1:]
local_indices = indices[:, 1:]
rows = np.repeat(np.arange(n_samples), local_distances.shape[1])
cols = local_indices.reshape(-1)
raw_d = local_distances.reshape(-1)

# Scale raw distances into [0, 1]
scale = np.percentile(raw_d, fairness_cfg["distance_scale_percentile"]) if raw_d.size else 1.0
scale = float(scale) if scale > 0 else 1.0
d = np.clip(raw_d / scale, 0.0, 1.0)

# Optional local threshold: only compare sufficiently similar individuals
if fairness_cfg["tau"] is not None:
    keep = d <= fairness_cfg["tau"]
    rows, cols, d = rows[keep], cols[keep], d[keep]

# 3) Fairness violations: max(0, |p_i - p_j| - d_ij)
delta_p = np.abs(p[rows] - p[cols])
violations = np.maximum(0.0, delta_p - d)
violation_mask = violations > 0

# 4) Optional Lipschitz ratios for d > 0
ratio_mask = d > 0
ratios = np.array([])
if np.any(ratio_mask):
    ratios = delta_p[ratio_mask] / d[ratio_mask]

print("Individual fairness (local) summary")
print("- evaluated pairs:", int(len(d)))
print("- distance scale (p{}): {:.4f}".format(fairness_cfg["distance_scale_percentile"], scale))
print("- tau:", fairness_cfg["tau"])
print("- max_violation:", round(float(violations.max()) if violations.size else 0.0, 6))
print("- avg_violation:", round(float(violations.mean()) if violations.size else 0.0, 6))
print("- violation_rate:", round(float(violation_mask.mean()) if violations.size else 0.0, 6))

if ratios.size:
    print("- max_lipschitz_ratio:", round(float(ratios.max()), 6))
    print("- p95_lipschitz_ratio:", round(float(np.percentile(ratios, 95)), 6))
    print("- ratio_gt_1_rate:", round(float((ratios > 1).mean()), 6))
else:
    print("- lipschitz ratios unavailable (all d_ij == 0)")


Individual fairness (local) summary
- evaluated pairs: 31653
- distance scale (p95.0): 2.1937
- tau: 0.35
- max_violation: 0.320421
- avg_violation: 0.003426
- violation_rate: 0.062459
- max_lipschitz_ratio: 4.799002
- p95_lipschitz_ratio: 1.089419
- ratio_gt_1_rate: 0.062466
